In [40]:
from email import policy
from email.parser import BytesParser
from email.utils import parseaddr, parsedate_to_datetime, getaddresses

from html import unescape
from html.parser import HTMLParser

from collections import defaultdict

from bs4 import BeautifulSoup
from pathlib import Path
import re

import pandas as pd
import glob
import os

In [42]:
INPUT_PATH = Path("../data/raw/emails.csv")
OUTPUT_PATH = Path("../data/processed/threads.csv")

emails = pd.read_csv(INPUT_PATH, index_col=False)

In [43]:
emails["date"] = pd.to_datetime(
    emails["date"],
    errors="coerce",
    utc=True
)

In [44]:
emails.columns

Index(['email_id', 'source_folder', 'folder', 'source_file', 'date',
       'direction', 'sender_name', 'sender_email', 'recipient_name',
       'recipient_email', 'cc', 'subject', 'body_raw', 'body_clean',
       'in_reply_to', 'references', 'is_reply', 'has_attachments'],
      dtype='str')

In [45]:
def extract_message_ids(value):
    """
    Extracts Message-IDs from an email header.

    Handles headers containing one or multiple IDs.
    """

    if pd.isna(value) or not str(value).strip():
        return []

    return re.findall(r"<[^<>]+>", str(value))

emails["email_id"] = emails["email_id"].fillna("").astype(str).str.strip()

emails["in_reply_to_ids"] = emails["in_reply_to"].apply(
    extract_message_ids
)

emails["reference_ids"] = emails["references"].apply(
    extract_message_ids
)
emails["direction"] = emails["direction"].astype(str).str.strip().str.lower()

In [46]:
class UnionFind:
    def __init__(self):
        self.parent = {}

    def add(self, item):
        if item not in self.parent:
            self.parent[item] = item

    def find(self, item):
        if item not in self.parent:
            self.add(item)

        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])

        return self.parent[item]

    def union(self, item1, item2):
        root1 = self.find(item1)
        root2 = self.find(item2)

        if root1 != root2:
            self.parent[root2] = root1

In [47]:
uf = UnionFind()
node_ids = []
for index, row in emails.iterrows():

    email_id = row["email_id"]

    if email_id:
        node_id = email_id
    else:
        node_id = f"missing_message_id_{index}"

    node_ids.append(node_id)
    uf.add(node_id)

In [48]:
for index, row in emails.iterrows():

    current_id = node_ids[index]

    parent_ids = (
        row["in_reply_to_ids"] +
        row["reference_ids"]
    )

    for parent_id in parent_ids:
        uf.union(current_id, parent_id)


In [49]:
emails["thread_root"] = [
    uf.find(node_id) for node_id in node_ids
]

In [50]:
print(f"Identified {emails['thread_root'].nunique()} threads.")

Identified 295 threads.


In [51]:
def get_first_valid(series):
    """
    Returns the first non-empty value.
    """

    for value in series:
        if pd.notna(value) and str(value).strip():
            return value

    return None

In [52]:
def build_thread(thread):

    # Sort emails chronologically
    thread = thread.sort_values("date", na_position="last")

    sent = thread[thread["direction"] == "sent"]
    received = thread[thread["direction"] == "received"]

    # First and last emails
    first_email = thread.iloc[0]

    first_email_date = thread["date"].min()
    last_email_date = thread["date"].max()

    # Initial outbound email
    first_sent = sent.iloc[0] if not sent.empty else None

    # First reply received after the initial outbound email
    first_reply = None

    if first_sent is not None and pd.notna(first_sent["date"]):

        replies_after_outreach = received[
            received["date"] > first_sent["date"]
        ]

        if not replies_after_outreach.empty:
            first_reply = replies_after_outreach.iloc[0]

    # Response time
    response_time_hours = None
    first_reply_date = None

    if first_reply is not None:

        first_reply_date = first_reply["date"]

        response_time_hours = (
            first_reply_date - first_sent["date"]
        ).total_seconds() / 3600

    # Identify the prospect
    # Prefer the sender of an inbound email.
    # If no inbound email exists, use the first outbound recipient.
    prospect_email = None
    prospect_name = None

    if not received.empty:

        prospect_email = get_first_valid(received["sender_email"])
        prospect_name = get_first_valid(received["sender_name"])

    elif first_sent is not None:

        prospect_email = first_sent["recipient_email"]
        prospect_name = first_sent["recipient_name"]

    # Subject and initial email ID
    subject_initial = get_first_valid(thread["subject"])

    initial_email_id = get_first_valid(thread["email_id"])

    # Thread ID
    thread_id = initial_email_id

    if not thread_id:
        thread_id = f"thread_{thread.index[0]}"

    # Follow-ups: sent emails after the first outbound email
    followup_count = max(len(sent) - 1, 0)

    return {

        "thread_id": thread_id,

        "prospect_email": prospect_email,
        "prospect_name": prospect_name,

        "subject_initial": subject_initial,

        "first_email_date": first_email_date,
        "last_email_date": last_email_date,

        "initial_email_id": initial_email_id,

        "total_emails": len(thread),
        "sent_count": len(sent),
        "received_count": len(received),

        "followup_count": followup_count,

        "first_reply_date": first_reply_date,
        "response_time_hours": response_time_hours,

        "replied": first_reply is not None
    }

In [53]:
thread_records = []

for _, group in emails.groupby("thread_root", sort=False):
    thread_records.append(build_thread(group))

threads = pd.DataFrame(thread_records)

print(f"Created {len(threads)} threads.")

threads.head()

Created 295 threads.


,thread_id,prospect_email,prospect_name,subject_initial,first_email_date,last_email_date,initial_email_id,total_emails,sent_count,received_count,followup_count,first_reply_date,response_time_hours,replied
0,<20261102223620.5dcee111cf4cfc18@cio54818.mail...,hello@mail.apollo.io,Apollo,Biggest sale of the year: Save $480 on the Pro...,2026-11-02 22:36:20+00:00,2026-11-02 22:36:20+00:00,<20261102223620.5dcee111cf4cfc18@cio54818.mail...,1,0,1,0,NaT,NaN,False
1,<1793659126727359347.1793659126@ibbagency.com>,support@saily.com,Saily,Saily on YouTube,2026-11-02 22:38:54+00:00,2026-11-04 19:02:25+00:00,<1793659126727359347.1793659126@ibbagency.com>,6,2,4,1,2026-11-02 22:40:06+00:00,0.020000,True
2,<010001a19e740aaf-b640e487-17a5-4be9-8823-2c81...,mailer@documentesign.com,Document eSign,Join request for Influencer Brand Brokerage Ag...,2026-11-02 22:49:27+00:00,2026-11-02 22:49:27+00:00,<010001a19e740aaf-b640e487-17a5-4be9-8823-2c81...,1,0,1,0,NaT,NaN,False
3,<Mime4j.0.a27a8c64832076eb.1a10d961e77@imap.gm...,operations@elitealliance.net,EA Ops,Sponsorship idea for Abyssal Detective,2026-10-05 19:41:44+00:00,2026-11-04 11:35:09+00:00,<Mime4j.0.a27a8c64832076eb.1a10d961e77@imap.gm...,10,6,4,5,2026-11-03 01:09:46+00:00,677.467222,True
4,<LNG71ZMGNKNN_6aea52f3ed109_05028ef03d080a_spr...,support@ultraloq.com,ULTRALOQ Support,[Request received] New customer message on Nov...,2026-11-03 21:32:36+00:00,2026-11-03 21:32:36+00:00,<LNG71ZMGNKNN_6aea52f3ed109_05028ef03d080a_spr...,1,0,1,0,NaT,NaN,False


In [54]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

threads.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(f"Exported threads to: {OUTPUT_PATH}")

Exported threads to: ../data/processed/threads.csv


In [55]:
# Categorize threads based on email direction

threads["thread_type"] = "Other"

threads.loc[
    (threads["sent_count"] > 0) &
    (threads["received_count"] == 0),
    "thread_type"
] = "Outbound only"

threads.loc[
    (threads["sent_count"] == 0) &
    (threads["received_count"] > 0),
    "thread_type"
] = "Inbound only"

threads.loc[
    (threads["sent_count"] > 0) &
    (threads["received_count"] > 0),
    "thread_type"
] = "Two-way conversation"


# Display distribution

print(threads["thread_type"].value_counts())

# Reply statistics by thread type

print("\nReply statistics:")
print(
    threads.groupby("thread_type")["replied"]
    .agg(["count", "sum", "mean"])
)

thread_type
Inbound only            163
Outbound only            93
Two-way conversation     39
Name: count, dtype: int64

Reply statistics:
                      count  sum      mean
thread_type                               
Inbound only            163    0  0.000000
Outbound only            93    0  0.000000
Two-way conversation     39   36  0.923077


In [56]:
threads[threads['prospect_email'].str.contains("stranger")]

,thread_id,prospect_email,prospect_name,subject_initial,first_email_date,last_email_date,initial_email_id,total_emails,sent_count,received_count,followup_count,first_reply_date,response_time_hours,replied,thread_type
61,<1789579993340027586.1789579993@ibbagency.com>,contactstrangerstories@gmail.com,Stranger Stories,Quick Update on the Sponsorship,2026-09-16 17:33:22+00:00,2026-10-20 17:56:49+00:00,<1789579993340027586.1789579993@ibbagency.com>,18,10,8,9,2026-09-16 17:35:02+00:00,0.027778,True,Two-way conversation
268,<1787864662685230554.1787864662@ibbagency.com>,contactstrangerstories@gmail.com,Stranger Stories,Sponsorship Opportunity for Stranger Stories.,2026-08-27 21:04:30+00:00,2026-08-30 15:05:41+00:00,<1787864662685230554.1787864662@ibbagency.com>,8,5,3,4,2026-08-27 21:59:20+00:00,0.913889,True,Two-way conversation
